In [1]:
import pandas as pd
import numpy as np

chi_train = pd.read_csv("chicago_2015_2024_model_ready.csv", low_memory=False)
chi_test  = pd.read_csv("chicago_2025_model_ready.csv", low_memory=False)

print("chi_train:", chi_train.shape)
print("chi_test :", chi_test.shape)
print(chi_train.columns.tolist())

chi_train: (311140, 45)
chi_test : (34424, 45)
['grid_id', 'iso_year', 'iso_week', 'year_week', 'total_crimes', 'count_THEFT', 'count_BATTERY', 'count_CRIMINAL DAMAGE', 'label_total_crime', 'label_THEFT', 'label_BATTERY', 'label_CRIMINAL DAMAGE', 'sin_week', 'cos_week', 'sin_month', 'cos_month', 'year_trend', 'lag_1w_total', 'lag_2w_total', 'lag_4w_total', 'lag_8w_total', 'lag_13w_total', 'lag_26w_total', 'lag_52w_total', 'roll_4w_total', 'roll_12w_total', 'roll_26w_total', 'neighbors_lag1w_sum', 'grid_lat', 'grid_lon', 'lag_1w_THEFT', 'lag_4w_THEFT', 'lag_13w_THEFT', 'lag_52w_THEFT', 'roll_4w_THEFT', 'lag_1w_BATTERY', 'lag_4w_BATTERY', 'lag_13w_BATTERY', 'lag_52w_BATTERY', 'roll_4w_BATTERY', 'lag_1w_CRIMINAL DAMAGE', 'lag_4w_CRIMINAL DAMAGE', 'lag_13w_CRIMINAL DAMAGE', 'lag_52w_CRIMINAL DAMAGE', 'roll_4w_CRIMINAL DAMAGE']


In [2]:
# Standardise the listing by replacing the spaced version 'CRIMINAL DAMAGE' with the underscored variant.
rename_map = {
    "count_CRIMINAL DAMAGE": "count_CRIMINAL_DAMAGE",
    "label_CRIMINAL DAMAGE": "label_CRIMINAL_DAMAGE",
    "lag_1w_CRIMINAL DAMAGE": "lag_1w_CRIMINAL_DAMAGE",
    "lag_4w_CRIMINAL DAMAGE": "lag_4w_CRIMINAL_DAMAGE",
    "lag_13w_CRIMINAL DAMAGE": "lag_13w_CRIMINAL_DAMAGE",
    "lag_52w_CRIMINAL DAMAGE": "lag_52w_CRIMINAL_DAMAGE",
    "roll_4w_CRIMINAL DAMAGE": "roll_4w_CRIMINAL_DAMAGE"
}

chi_train = chi_train.rename(columns=rename_map)
chi_test  = chi_test.rename(columns=rename_map)

print("Renamed done.")

Renamed done.


In [3]:
# Combine the train and test datasets to create shared features.
chi_all = pd.concat([chi_train, chi_test], ignore_index=True)

chi_all = chi_all.sort_values(
    ["grid_id", "iso_year", "iso_week"]
).reset_index(drop=True)

print("chi_all:", chi_all.shape)

chi_all: (345564, 45)


In [4]:
# Fill in the missing shared lag feature
lag_steps_extra = [2, 8, 26]
crime_count_map = {
    "THEFT": "count_THEFT",
    "BATTERY": "count_BATTERY",
    "CRIMINAL_DAMAGE": "count_CRIMINAL_DAMAGE"
}

for lag in lag_steps_extra:
    for crime_name, count_col in crime_count_map.items():
        new_col = f"lag_{lag}w_{crime_name}"
        chi_all[new_col] = (
            chi_all.groupby("grid_id")[count_col].shift(lag)
        )

In [5]:
check_cols = [
    "lag_2w_THEFT", "lag_8w_THEFT", "lag_26w_THEFT",
    "lag_2w_BATTERY", "lag_8w_BATTERY", "lag_26w_BATTERY",
    "lag_2w_CRIMINAL_DAMAGE", "lag_8w_CRIMINAL_DAMAGE", "lag_26w_CRIMINAL_DAMAGE"
]
print(chi_all[check_cols].head(10))

   lag_2w_THEFT  lag_8w_THEFT  lag_26w_THEFT  lag_2w_BATTERY  lag_8w_BATTERY  \
0           NaN           NaN            NaN             NaN             NaN   
1           NaN           NaN            NaN             NaN             NaN   
2           0.0           NaN            NaN             1.0             NaN   
3           0.0           NaN            NaN             0.0             NaN   
4           0.0           NaN            NaN             0.0             NaN   
5           0.0           NaN            NaN             0.0             NaN   
6           0.0           NaN            NaN             2.0             NaN   
7           0.0           NaN            NaN             0.0             NaN   
8           0.0           0.0            NaN             1.0             1.0   
9           1.0           0.0            NaN             0.0             0.0   

   lag_26w_BATTERY  lag_2w_CRIMINAL_DAMAGE  lag_8w_CRIMINAL_DAMAGE  \
0              NaN                     NaN       

In [6]:
# Delete the Chicago-specific, non-shared columns
drop_cols = [
    "sin_month",
    "cos_month",
    "year_trend",
    "neighbors_lag1w_sum",
    "grid_lat",
    "grid_lon"
]

chi_all = chi_all.drop(columns=drop_cols, errors="ignore")

In [7]:
# Uniform handling of missing values
lag_cols_chi = [c for c in chi_all.columns if c.startswith("lag_")]
roll_cols_chi = [c for c in chi_all.columns if c.startswith("roll_")]

chi_all[lag_cols_chi + roll_cols_chi] = chi_all[lag_cols_chi + roll_cols_chi].fillna(0)

In [8]:
# Define the final shared attribute columns (42 columns)
shared_feature_cols = [
    # total
    "lag_1w_total", "lag_2w_total", "lag_4w_total", "lag_8w_total", "lag_13w_total", "lag_26w_total", "lag_52w_total",
    "roll_4w_total", "roll_12w_total", "roll_26w_total",

    # THEFT
    "lag_1w_THEFT", "lag_2w_THEFT", "lag_4w_THEFT", "lag_8w_THEFT", "lag_13w_THEFT", "lag_26w_THEFT", "lag_52w_THEFT",
    "roll_4w_THEFT", "roll_12w_THEFT", "roll_26w_THEFT",

    # BATTERY
    "lag_1w_BATTERY", "lag_2w_BATTERY", "lag_4w_BATTERY", "lag_8w_BATTERY", "lag_13w_BATTERY", "lag_26w_BATTERY", "lag_52w_BATTERY",
    "roll_4w_BATTERY", "roll_12w_BATTERY", "roll_26w_BATTERY",

    # CRIMINAL_DAMAGE
    "lag_1w_CRIMINAL_DAMAGE", "lag_2w_CRIMINAL_DAMAGE", "lag_4w_CRIMINAL_DAMAGE", "lag_8w_CRIMINAL_DAMAGE",
    "lag_13w_CRIMINAL_DAMAGE", "lag_26w_CRIMINAL_DAMAGE", "lag_52w_CRIMINAL_DAMAGE",
    "roll_4w_CRIMINAL_DAMAGE", "roll_12w_CRIMINAL_DAMAGE", "roll_26w_CRIMINAL_DAMAGE",

    # time
    "sin_week", "cos_week"
]

print("Shared feature count:", len(shared_feature_cols))

Shared feature count: 42


In [10]:
rolling_windows_extra = [12, 26]

crime_count_map = {
    "THEFT": "count_THEFT",
    "BATTERY": "count_BATTERY",
    "CRIMINAL_DAMAGE": "count_CRIMINAL_DAMAGE"
}

for window in rolling_windows_extra:
    for crime_name, count_col in crime_count_map.items():
        
        new_col = f"roll_{window}w_{crime_name}"
        
        chi_all[new_col] = (
            chi_all
            .groupby("grid_id")[count_col]
            .shift(1)                     # 防止数据泄露
            .rolling(window)
            .mean()
            .reset_index(level=0, drop=True)
        )

print("Added rolling crime features.")

Added rolling crime features.


In [11]:
lag_cols = [c for c in chi_all.columns if c.startswith("lag_")]
roll_cols = [c for c in chi_all.columns if c.startswith("roll_")]

chi_all[lag_cols + roll_cols] = chi_all[lag_cols + roll_cols].fillna(0)

In [12]:
missing_in_chicago = [c for c in shared_feature_cols if c not in chi_all.columns]
print("Missing in Chicago:", missing_in_chicago)

Missing in Chicago: []


In [13]:
# Split the Chicago dataset into train and test sets by year
chi_train_shared = chi_all[chi_all["iso_year"] <= 2024].copy()
chi_test_shared  = chi_all[chi_all["iso_year"] >= 2025].copy()

print("chi_train_shared:", chi_train_shared.shape)
print("chi_test_shared :", chi_test_shared.shape)

chi_train_shared: (311140, 54)
chi_test_shared : (34424, 54)


In [14]:
# Define the columns to be retained in the final output.
id_cols = ["grid_id", "iso_year", "iso_week", "year_week"]

target_count_cols = [
    "total_crimes",
    "count_THEFT",
    "count_BATTERY",
    "count_CRIMINAL_DAMAGE"
]

label_cols = [
    "label_total_crime",
    "label_THEFT",
    "label_BATTERY",
    "label_CRIMINAL_DAMAGE"
]

In [15]:
# Construct a Chicago shared final table
chi_train_shared_final = chi_train_shared[
    id_cols + target_count_cols + label_cols + shared_feature_cols
].copy()

chi_test_shared_final = chi_test_shared[
    id_cols + target_count_cols + label_cols + shared_feature_cols
].copy()

print("chi_train_shared_final:", chi_train_shared_final.shape)
print("chi_test_shared_final :", chi_test_shared_final.shape)

chi_train_shared_final: (311140, 54)
chi_test_shared_final : (34424, 54)


In [16]:
# Construct the feature matrix of the pure model
X_chi_train_shared = chi_train_shared_final[shared_feature_cols].copy()
X_chi_test_shared  = chi_test_shared_final[shared_feature_cols].copy()

print("X_chi_train_shared:", X_chi_train_shared.shape)
print("X_chi_test_shared :", X_chi_test_shared.shape)

X_chi_train_shared: (311140, 42)
X_chi_test_shared : (34424, 42)


In [17]:
print(shared_feature_cols[:10])
print(len(shared_feature_cols))

['lag_1w_total', 'lag_2w_total', 'lag_4w_total', 'lag_8w_total', 'lag_13w_total', 'lag_26w_total', 'lag_52w_total', 'roll_4w_total', 'roll_12w_total', 'roll_26w_total']
42


In [18]:
chi_train_shared_final.to_csv("chicago_2015_2024_shared_final.csv", index=False)
chi_test_shared_final.to_csv("chicago_2025_shared_final.csv", index=False)

X_chi_train_shared.to_csv("chicago_2015_2024_X_shared_42cols.csv", index=False)
X_chi_test_shared.to_csv("chicago_2025_X_shared_42cols.csv", index=False)

print("Saved:")
print(" - chicago_2015_2024_shared_final.csv", chi_train_shared_final.shape)
print(" - chicago_2025_shared_final.csv", chi_test_shared_final.shape)
print(" - chicago_2015_2024_X_shared_42cols.csv", X_chi_train_shared.shape)
print(" - chicago_2025_X_shared_42cols.csv", X_chi_test_shared.shape)

Saved:
 - chicago_2015_2024_shared_final.csv (311140, 54)
 - chicago_2025_shared_final.csv (34424, 54)
 - chicago_2015_2024_X_shared_42cols.csv (311140, 42)
 - chicago_2025_X_shared_42cols.csv (34424, 42)
